# Basin-Specific 3-Model Ensemble — NetCDF Generation
## Full Period: 1961–2070 | SSP2-4.5

**Purpose:**  
For each Jordan hydrological basin, construct a **3-model ensemble** NetCDF file covering the full 1961–2070 period. The three models contributing to each basin's ensemble are the top-ranked models identified during the validation step (Notebook 03), following the composite ranking system (Equation 5, Hussien et al.).

**Spatial logic:**  
- Basin boundaries are read from the Jordan surface basins shapefile (`Jo.shp`)  
- Only Jordan-domain basins are processed; Syrian sub-basins are excluded  
- Basins without station-based validation data (e.g. WADI ARABA SOUTH, QA DISI & SOUTHERN DESERT, SARHAN, J.VALLEY-YARMOUK TRIANGLE) inherit the top-3 model assignment from the nearest basin with validation data, using a k-d tree on basin centroids — consistent with Section 2.5 of the manuscript  
- Each grid cell within a basin receives the ensemble mean of the three assigned models

**Ensemble construction:**  
$$P_{\text{ens3}}(i,j,t) = \frac{1}{3} \sum_{k=1}^{3} M_k(i,j,t)$$

where $M_k$ is daily precipitation (mm/day) from the $k$-th ranked model, and $(i,j)$ is a grid point within the basin polygon.

**Inputs:**  
- `basin_model_rankings.csv` (from Notebook 03)  
- `Jo.shp` — Jordan surface basin shapefile (GCS_WGS_1984)  
- Six model NetCDF files (1961–2070, variable `prAdjust`, mm/day)

**Output:**  
One NetCDF file per Jordan basin → `C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\3 ensemble models\`  
Filename format: `<BASIN_SAFE_NAME>_3model_ensemble_ssp245.nc`


## 1. Import Libraries

In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree
from shapely.geometry import box
import datetime

warnings.filterwarnings("ignore")

print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr), ("geopandas", gpd)]:
    print(f"  {lib:<12}: {mod.__version__}")


Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0
  geopandas   : 1.0.1


## 2. Configuration

All paths and parameters are defined here.


In [2]:
# ── Input paths ───────────────────────────────────────────────────────────────
SHAPEFILE     = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                     r"\jordan.basins\surface.basins.for.jordan\Jo.shp")

RANKINGS_CSV  = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                     r"\validation\single.model\basin_model_rankings.csv")

MODEL_DIR     = Path(r"D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915"
                     r"\Merge\Precipitation")

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR    = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                     r"\3 ensemble models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model list ────────────────────────────────────────────────────────────────
MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]

PR_VAR = "prAdjust"   # precipitation variable in NetCDF files (mm/day)

# ── Basins to EXCLUDE (Syrian portions of transboundary basins) ────────────
SYRIA_BASINS = {
    "YARMOUK (SYRIA)",
    "AZRAQ (SYRIA)",
    "AMMAN ZARQA (SYRIA)",
}

# ── Shapefile BASIN_NAME  →  validation ranking basin name ───────────────────
# Basins present in Jo.shp that have direct validation rankings keep their name.
# Basins without station data are mapped to the nearest ranked basin via
# centroid k-d tree (see Cell 5). This dict only covers explicit name
# corrections needed between shapefile spelling and ranking CSV spelling.
BASIN_NAME_MAP = {
    # shapefile name              : ranking CSV name (None = use KD-tree)
    "HAMMAD"                     : "HAMMAD",
    "YARMOUK (JORDAN)"           : "YARMOUK (JORDAN)",
    "J.VALLEY-YARMOUK TRIANGLE"  : None,          # no stations → KD-tree
    "JORDAN VALLY (JORDAN)"      : "JORDAN VALLY (JORDAN)",
    "N.R.S.W"                    : "N.R.S.W",
    "AZRAQ (JORDAN)"             : "AZRAQ (JORDAN)",
    "AMMAN ZARQA (JORDAN)"       : "AMMAN ZARQA (JORDAN)",
    "S.R.S.W"                    : "S.R.S.W",
    "MUJIB"                      : "MUJIB",
    "D.S.R.S.W"                  : "D.S.R.S.W",
    "W. ARABA NORTH"             : "W. ARABA NORTH",
    "HASA"                       : "HASA",
    "JAFER"                      : "JAFER",
    "WADI ARABA SOUTH"           : None,           # no stations → KD-tree
    "QA DISI & SOUTHERN DESERT"  : None,           # no stations → KD-tree
    "SARHAN"                     : None,            # no stations → KD-tree
}

print("Configuration loaded.")
print(f"  Shapefile  : {SHAPEFILE}")
print(f"  Rankings   : {RANKINGS_CSV}")
print(f"  Model dir  : {MODEL_DIR}")
print(f"  Output dir : {OUTPUT_DIR}")


Configuration loaded.
  Shapefile  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\jordan.basins\surface.basins.for.jordan\Jo.shp
  Rankings   : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\single.model\basin_model_rankings.csv
  Model dir  : D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915\Merge\Precipitation
  Output dir : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\3 ensemble models


## 3. Load Basin Shapefile

The Jordan surface basin shapefile is loaded and filtered to retain only Jordan-domain polygons (Syrian portions are excluded). Basin names are standardised against the `BASIN_NAME_MAP` dictionary.


In [3]:
gdf = gpd.read_file(SHAPEFILE)
print(f"Total features in shapefile : {len(gdf)}")
print(f"CRS                         : {gdf.crs}")
print(f"Columns                     : {gdf.columns.tolist()}")
print()
print("All BASIN_NAME values found:")
for name in gdf["BASIN_NAME"].tolist():
    print(f"  {name}")


Total features in shapefile : 19
CRS                         : EPSG:4326
Columns                     : ['AREA', 'PERIMETER', 'MAINSWBASI', 'IDEN', 'BASIN_NAME', 'NWMP_CODE', 'dis', 'geometry']

All BASIN_NAME values found:
  HAMMAD
  YARMOUK (SYRIA)
  YARMOUK (JORDAN)
  J.VALLEY-YARMOUK TRIANGLE
  JORDAN VALLY (JORDAN)
  N.R.S.W
  AZRAQ (SYRIA)
  AMMAN ZARQA (SYRIA)
  AZRAQ (JORDAN)
  AMMAN ZARQA (JORDAN)
  S.R.S.W
  MUJIB
  D.S.R.S.W
  W. ARABA NORTH
  HASA
  JAFER
  WADI ARABA SOUTH
  QA DISI & SOUTHERN DESERT
  SARHAN


In [4]:
# ── Filter out Syria basins ───────────────────────────────────────────────────
gdf_jordan = gdf[~gdf["BASIN_NAME"].isin(SYRIA_BASINS)].copy()
gdf_jordan = gdf_jordan.reset_index(drop=True)

print(f"Jordan-only basins retained : {len(gdf_jordan)}")
print()
print("Retained basins:")
for _, row in gdf_jordan.iterrows():
    print(f"  FID {row.name:>2}  |  {row['BASIN_NAME']}")


Jordan-only basins retained : 16

Retained basins:
  FID  0  |  HAMMAD
  FID  1  |  YARMOUK (JORDAN)
  FID  2  |  J.VALLEY-YARMOUK TRIANGLE
  FID  3  |  JORDAN VALLY (JORDAN)
  FID  4  |  N.R.S.W
  FID  5  |  AZRAQ (JORDAN)
  FID  6  |  AMMAN ZARQA (JORDAN)
  FID  7  |  S.R.S.W
  FID  8  |  MUJIB
  FID  9  |  D.S.R.S.W
  FID 10  |  W. ARABA NORTH
  FID 11  |  HASA
  FID 12  |  JAFER
  FID 13  |  WADI ARABA SOUTH
  FID 14  |  QA DISI & SOUTHERN DESERT
  FID 15  |  SARHAN


## 4. Load Validation Rankings

The composite ranking results from Notebook 03 are loaded. For each basin that has validation data, the top-3 ranked models are extracted.


In [5]:
rankings = pd.read_csv(RANKINGS_CSV)
print(f"Rankings loaded : {len(rankings)} rows")
print(f"Columns         : {rankings.columns.tolist()}")
print()

# Extract top-3 per ranked basin (Basin_Rank column, lowest = best)
top3_per_basin = {}
for basin, grp in rankings.groupby("Basin"):
    grp_sorted = grp.sort_values("Avg_Rank")
    top3 = grp_sorted["Model"].iloc[:3].tolist()
    top3_per_basin[basin] = top3

print("Top-3 models per validated basin:")
print(f"{'Basin':<28} {'Rank-1':<22} {'Rank-2':<22} {'Rank-3'}")
print("-" * 85)
for basin, models in top3_per_basin.items():
    m1 = models[0] if len(models) > 0 else "-"
    m2 = models[1] if len(models) > 1 else "-"
    m3 = models[2] if len(models) > 2 else "-"
    print(f"{basin:<28} {m1:<22} {m2:<22} {m3}")


Rankings loaded : 72 rows
Columns         : ['Model', 'r', 'NSE', 'PBIAS', 'RMSE', 'MAE', 'Rank_r', 'Rank_NSE', 'Rank_absPBIAS', 'Rank_RMSE', 'Rank_MAE', 'Avg_Rank', 'Basin', 'Basin_Rank', 'N_Stations']

Top-3 models per validated basin:
Basin                        Rank-1                 Rank-2                 Rank-3
-------------------------------------------------------------------------------------
AMMAN ZARQA (JORDAN)         MPI-ESM1-2-LR          NorESM2-MM             CMCC-CM2-SR5
AZRAQ (JORDAN)               MPI-ESM1-2-LR          NorESM2-MM             CMCC-CM2-SR5
D.S.R.S.W                    IPSL-CM6A-LR           MPI-ESM1-2-LR          NorESM2-MM
HAMMAD                       MPI-ESM1-2-LR          CMCC-CM2-SR5           NorESM2-MM
HASA                         MPI-ESM1-2-LR          CMCC-CM2-SR5           CNRM-ESM2-1
JAFER                        NorESM2-MM             MPI-ESM1-2-LR          CMCC-CM2-SR5
JORDAN VALLY (JORDAN)        MPI-ESM1-2-LR          CMCC-CM2-SR5       

## 5. Assign Top-3 Models to All Jordan Basins

Basins without station-based validation data (J.VALLEY-YARMOUK TRIANGLE, WADI ARABA SOUTH, QA DISI & SOUTHERN DESERT, SARHAN) inherit the top-3 model assignment from their nearest validated basin, measured by distance between basin centroids. This approach is consistent with the KD-tree nearest-basin method described in Section 2.5 of the manuscript (Bentley, 1975).


In [6]:
# Compute centroids in geographic coordinates
gdf_jordan["centroid_lon"] = gdf_jordan.geometry.centroid.x
gdf_jordan["centroid_lat"] = gdf_jordan.geometry.centroid.y

# Ranked basins: those in BASIN_NAME_MAP with a direct ranking match
ranked_basins_gdf = gdf_jordan[
    gdf_jordan["BASIN_NAME"].apply(
        lambda n: BASIN_NAME_MAP.get(n) is not None and
                  BASIN_NAME_MAP.get(n) in top3_per_basin
    )
].copy()

unranked_basins_gdf = gdf_jordan[
    ~gdf_jordan.index.isin(ranked_basins_gdf.index)
].copy()

print(f"Basins with direct ranking data : {len(ranked_basins_gdf)}")
print(f"Basins requiring KD-tree match  : {len(unranked_basins_gdf)}")
print()

# Build KD-tree on ranked basin centroids
ranked_coords = ranked_basins_gdf[["centroid_lat","centroid_lon"]].values
tree = cKDTree(ranked_coords)

# Assign top-3 models to every Jordan basin
basin_top3_assignment = {}

for _, row in gdf_jordan.iterrows():
    shp_name      = row["BASIN_NAME"]
    ranking_name  = BASIN_NAME_MAP.get(shp_name)

    if ranking_name is not None and ranking_name in top3_per_basin:
        # Direct match
        top3 = top3_per_basin[ranking_name]
        source = "direct"
    else:
        # KD-tree: find nearest ranked basin centroid
        query_pt   = [row["centroid_lat"], row["centroid_lon"]]
        dist, idx  = tree.query(query_pt)
        nearest_row = ranked_basins_gdf.iloc[idx]
        nearest_shp = nearest_row["BASIN_NAME"]
        nearest_rank= BASIN_NAME_MAP[nearest_shp]
        top3        = top3_per_basin[nearest_rank]
        source      = f"KD-tree → {nearest_shp} ({dist*111.32:.1f} km)"

    basin_top3_assignment[shp_name] = {
        "top3"  : top3,
        "source": source,
    }

print(f"{'Basin':<30} {'Source':<35} {'Top-3 Models'}")
print("-" * 100)
for basin, info in basin_top3_assignment.items():
    print(f"{basin:<30} {info['source']:<35} {' | '.join(info['top3'])}")


Basins with direct ranking data : 12
Basins requiring KD-tree match  : 4

Basin                          Source                              Top-3 Models
----------------------------------------------------------------------------------------------------
HAMMAD                         direct                              MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
YARMOUK (JORDAN)               direct                              MPI-ESM1-2-LR | NorESM2-MM | CNRM-ESM2-1
J.VALLEY-YARMOUK TRIANGLE      KD-tree → N.R.S.W (29.2 km)         NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
JORDAN VALLY (JORDAN)          direct                              MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
N.R.S.W                        direct                              NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
AZRAQ (JORDAN)                 direct                              MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
AMMAN ZARQA (JORDAN)           direct                              MPI-ESM1-2-LR | NorESM2-MM | 

## 6. Build Grid Cell – Basin Mask

For each Jordan basin polygon, identify all 0.1° grid cells whose **centres** fall within (or intersect) the basin boundary. The mask uses `shapely.geometry.box` to construct each grid cell as a small polygon and checks intersection with the basin geometry — consistent with the spatial masking approach described in the manuscript (using `.intersects()` rather than centre-point containment, to avoid excluding boundary grid cells).


In [7]:
# Open one model to get the grid (all models share the same grid)
sample_nc = MODEL_DIR / f"{MODELS[0]}.nc"
with xr.open_dataset(sample_nc) as ds:
    lats = ds["lat"].values   # shape (85,)
    lons = ds["lon"].values   # shape (75,)
    # Store time info for later
    sample_times = ds["time"].values
    sample_attrs = dict(ds[PR_VAR].attrs)
    global_attrs = dict(ds.attrs)

print(f"Model grid: {len(lats)} lat × {len(lons)} lon  "
      f"({lats.min():.2f}–{lats.max():.2f}°N, {lons.min():.2f}–{lons.max():.2f}°E)")
print(f"Full period time steps: {len(sample_times)}  "
      f"({pd.Timestamp(sample_times[0]).date()} → {pd.Timestamp(sample_times[-1]).date()})")
print()

# Half-cell size for intersection boxes
half = 0.05   # 0.1° / 2

# Build mask for every Jordan basin
basin_masks = {}   # basin_name → dict with lat_indices, lon_indices, 2D bool mask

for _, brow in gdf_jordan.iterrows():
    bname  = brow["BASIN_NAME"]
    bgeom  = brow["geometry"]

    lat_idxs = []
    lon_idxs = []

    for li, lat in enumerate(lats):
        for lj, lon in enumerate(lons):
            cell_box = box(lon - half, lat - half, lon + half, lat + half)
            if bgeom.intersects(cell_box):
                lat_idxs.append(li)
                lon_idxs.append(lj)

    basin_masks[bname] = {
        "lat_idxs": np.array(lat_idxs),
        "lon_idxs": np.array(lon_idxs),
        "n_cells" : len(lat_idxs),
    }
    print(f"  {bname:<32} : {len(lat_idxs):>4} grid cells")

print()
total_cells = sum(v["n_cells"] for v in basin_masks.values())
print(f"Total grid cell assignments  : {total_cells}")
print()

# Warn if any basin has zero grid cells
empty_basins = [b for b, v in basin_masks.items() if v["n_cells"] == 0]
if empty_basins:
    print("WARNING — basins with ZERO grid cells (too small or outside domain):")
    for b in empty_basins:
        print(f"  {b}")


Model grid: 85 lat × 75 lon  (27.05–35.45°N, 33.55–40.95°E)
Full period time steps: 40177  (1961-01-01 → 2070-12-31)

  HAMMAD                           :  217 grid cells
  YARMOUK (JORDAN)                 :   26 grid cells
  J.VALLEY-YARMOUK TRIANGLE        :    3 grid cells
  JORDAN VALLY (JORDAN)            :   21 grid cells
  N.R.S.W                          :   16 grid cells
  AZRAQ (JORDAN)                   :  149 grid cells
  AMMAN ZARQA (JORDAN)             :   54 grid cells
  S.R.S.W                          :   16 grid cells
  MUJIB                            :   91 grid cells
  D.S.R.S.W                        :   31 grid cells
  W. ARABA NORTH                   :   45 grid cells
  HASA                             :   43 grid cells
  JAFER                            :  148 grid cells
  WADI ARABA SOUTH                 :   73 grid cells
  QA DISI & SOUTHERN DESERT        :   65 grid cells
  SARHAN                           :  198 grid cells

Total grid cell assignments  : 11

## 7. Filename Utilities


In [8]:
def safe_filename(basin_name: str) -> str:
    """
    Convert a basin name to a safe filename component.
    Spaces → underscores; remove characters invalid in Windows filenames.
    """
    s = basin_name.strip()
    s = re.sub(r"[\s]+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-\.]", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


# Preview filename mapping
print("Basin  →  filename mapping:")
print("-" * 60)
for bname in basin_top3_assignment:
    print(f"  {bname:<32} →  {safe_filename(bname)}_3model_ensemble_ssp245.nc")


Basin  →  filename mapping:
------------------------------------------------------------
  HAMMAD                           →  HAMMAD_3model_ensemble_ssp245.nc
  YARMOUK (JORDAN)                 →  YARMOUK_JORDAN_3model_ensemble_ssp245.nc
  J.VALLEY-YARMOUK TRIANGLE        →  J.VALLEY-YARMOUK_TRIANGLE_3model_ensemble_ssp245.nc
  JORDAN VALLY (JORDAN)            →  JORDAN_VALLY_JORDAN_3model_ensemble_ssp245.nc
  N.R.S.W                          →  N.R.S.W_3model_ensemble_ssp245.nc
  AZRAQ (JORDAN)                   →  AZRAQ_JORDAN_3model_ensemble_ssp245.nc
  AMMAN ZARQA (JORDAN)             →  AMMAN_ZARQA_JORDAN_3model_ensemble_ssp245.nc
  S.R.S.W                          →  S.R.S.W_3model_ensemble_ssp245.nc
  MUJIB                            →  MUJIB_3model_ensemble_ssp245.nc
  D.S.R.S.W                        →  D.S.R.S.W_3model_ensemble_ssp245.nc
  W. ARABA NORTH                   →  W._ARABA_NORTH_3model_ensemble_ssp245.nc
  HASA                             →  HASA_3model_ensemble_s

## 8. Generate Basin Ensemble NetCDF Files

For each Jordan basin:

1. Identify the three assigned models  
2. Open each model's NetCDF file  
3. For every grid cell within the basin, compute the pixel-wise arithmetic mean across the three models at every time step  
4. Assemble a new `xarray.Dataset` with the ensemble precipitation, preserving the original coordinate system, time dimension, and CF metadata  
5. Add provenance attributes documenting which models contributed and the basin assignment method  
6. Write to a compressed NetCDF4 file

> **Memory strategy:** Models are opened with `xr.open_dataset` (lazy loading). Only the basin's grid cells are extracted and loaded into memory, keeping peak RAM usage proportional to the basin size × 3 models, not the full grid.


In [9]:
import time as _time

# Pre-open all 6 model datasets lazily (closed in finally block)
model_datasets = {}
for model in MODELS:
    nc_path = MODEL_DIR / f"{model}.nc"
    model_datasets[model] = xr.open_dataset(nc_path)
    print(f"  Opened (lazy): {model}.nc")

print()
processed = []
failed    = []

for basin_name, assign_info in basin_top3_assignment.items():
    top3      = assign_info["top3"]
    mask_info = basin_masks.get(basin_name)

    if mask_info is None or mask_info["n_cells"] == 0:
        print(f"[SKIP] {basin_name} — no grid cells in domain.")
        failed.append({"Basin": basin_name, "Status": "SKIPPED – no grid cells"})
        continue

    lat_idxs = mask_info["lat_idxs"]
    lon_idxs = mask_info["lon_idxs"]

    out_filename = safe_filename(basin_name) + "_3model_ensemble_ssp245.nc"
    out_path     = OUTPUT_DIR / out_filename

    if out_path.exists():
        print(f"[SKIP] {basin_name} — output already exists.")
        processed.append({"Basin": basin_name, "Status": "EXISTS", "File": out_filename})
        continue

    t0 = _time.time()
    print(f"[PROCESSING] {basin_name}")
    print(f"  Models  : {' | '.join(top3)}")
    print(f"  Cells   : {mask_info['n_cells']}  |  Time steps: {len(sample_times):,}")

    try:
        # Get shared coordinates
        ds_ref   = model_datasets[top3[0]]
        time_arr = ds_ref["time"].values
        lat_vals = lats[np.unique(lat_idxs)]
        lon_vals = lons[np.unique(lon_idxs)]

        # Build a mapping from (lat_idx, lon_idx) → position in output grid
        # Output grid: subset of lat/lon spanning this basin's bounding box
        lat_unique = np.sort(np.unique(lat_idxs))
        lon_unique = np.sort(np.unique(lon_idxs))
        out_lats   = lats[lat_unique]
        out_lons   = lons[lon_unique]
        n_lat_out  = len(out_lats)
        n_lon_out  = len(out_lons)
        n_time     = len(time_arr)

        # Map global indices → local output indices
        lat_local = {gi: li for li, gi in enumerate(lat_unique)}
        lon_local = {gi: li for li, gi in enumerate(lon_unique)}

        # Allocate accumulator for ensemble sum
        # Shape: (time, n_lat_out, n_lon_out) — NaN outside basin
        ens_sum = np.full((n_time, n_lat_out, n_lon_out), np.nan, dtype=np.float32)
        n_models_contributing = np.zeros((n_lat_out, n_lon_out), dtype=np.int8)

        for model in top3:
            ds_m  = model_datasets[model]
            pr_m  = ds_m[PR_VAR]   # DataArray (time, lat, lon), lazy

            # Extract only the lat/lon slice covering this basin's bounding box
            pr_sub = pr_m.isel(
                lat=xr.DataArray(lat_idxs, dims="pts"),
                lon=xr.DataArray(lon_idxs, dims="pts")
            ).values   # loads into memory: shape (n_time, n_pts)
            # pr_sub shape: (n_time, n_pts)

            for pt_idx, (gi_lat, gi_lon) in enumerate(zip(lat_idxs, lon_idxs)):
                li = lat_local[gi_lat]
                lj = lon_local[gi_lon]
                col = pr_sub[:, pt_idx].astype(np.float32)
                # First model: initialise; subsequent: add
                if np.isnan(ens_sum[0, li, lj]):
                    ens_sum[:, li, lj]  = col
                else:
                    ens_sum[:, li, lj] += col
                n_models_contributing[li, lj] += 1

        # Divide by number of models contributing to each cell
        ens_pr = np.where(
            n_models_contributing[np.newaxis, :, :] > 0,
            ens_sum / n_models_contributing[np.newaxis, :, :],
            np.nan
        ).astype(np.float32)

        # ── Build output xarray Dataset ───────────────────────────────────────
        da = xr.DataArray(
            ens_pr,
            dims=["time", "lat", "lon"],
            coords={
                "time": time_arr,
                "lat" : out_lats,
                "lon" : out_lons,
            },
            name=PR_VAR,
            attrs={
                "standard_name"  : "precipitation_flux",
                "long_name"      : "3-Model Ensemble Mean Precipitation",
                "units"          : "mm/day",
                "cell_methods"   : "time: mean",
                "ensemble_models": " | ".join(top3),
                "ensemble_size"  : 3,
                "basin_name"     : basin_name,
                "assignment_source": assign_info["source"],
            }
        )

        ds_out = xr.Dataset(
            {PR_VAR: da},
            attrs={
                "title"          : f"3-Model Ensemble Precipitation — {basin_name}",
                "institution"    : "Jordan University of Science and Technology",
                "source_models"  : " | ".join(top3),
                "basin"          : basin_name,
                "assignment_method": assign_info["source"],
                "scenario"       : "SSP2-4.5",
                "domain"         : "MSH-10 (Jordan domain, 0.1° resolution)",
                "original_data"  : "RICCAR SMHI-HCLIM-ALADIN-38, bias-corrected via SMHI-MIdAS021",
                "time_coverage_start": str(pd.Timestamp(time_arr[0]).date()),
                "time_coverage_end"  : str(pd.Timestamp(time_arr[-1]).date()),
                "Conventions"    : "CF-1.7",
                "history"        : (f"Created {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} "
                                    f"using 3-model ensemble of top-ranked CMIP6 models per basin "
                                    f"(Hussien et al. evaluation framework)"),
            }
        )

        # Add lat/lon attributes
        ds_out["lat"].attrs = {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}
        ds_out["lon"].attrs = {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}

        # ── Write compressed NetCDF4 ──────────────────────────────────────────
        encoding = {
            PR_VAR: {
                "dtype"      : "float32",
                "zlib"       : True,
                "complevel"  : 4,
                "_FillValue" : np.float32(1e+20),
            }
        }
        ds_out.to_netcdf(out_path, encoding=encoding, format="NETCDF4")

        elapsed = _time.time() - t0
        file_mb = out_path.stat().st_size / 1024 / 1024
        print(f"  Saved  : {out_filename}  ({file_mb:.1f} MB)  [{elapsed:.0f}s]")
        processed.append({
            "Basin"  : basin_name,
            "Status" : "OK",
            "File"   : out_filename,
            "Size_MB": round(file_mb, 1),
            "Models" : " | ".join(top3),
            "N_Cells": mask_info["n_cells"],
        })

    except Exception as e:
        print(f"  ERROR  : {e}")
        failed.append({"Basin": basin_name, "Status": f"ERROR: {e}"})

    print()

# Close all datasets
for ds in model_datasets.values():
    ds.close()

print("=" * 70)
print("ENSEMBLE GENERATION COMPLETE")
print("=" * 70)
print(f"  Processed : {len([p for p in processed if p['Status']=='OK'])}")
print(f"  Skipped   : {len([p for p in processed if p['Status'] in ('EXISTS','SKIPPED – no grid cells')])}")
print(f"  Failed    : {len(failed)}")


  Opened (lazy): CMCC-CM2-SR5.nc
  Opened (lazy): CNRM-ESM2-1.nc
  Opened (lazy): EC-Earth3-Veg.nc
  Opened (lazy): IPSL-CM6A-LR.nc
  Opened (lazy): MPI-ESM1-2-LR.nc
  Opened (lazy): NorESM2-MM.nc

[PROCESSING] HAMMAD
  Models  : MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
  Cells   : 217  |  Time steps: 40,177
  Saved  : HAMMAD_3model_ensemble_ssp245.nc  (24.5 MB)  [8s]

[PROCESSING] YARMOUK (JORDAN)
  Models  : MPI-ESM1-2-LR | NorESM2-MM | CNRM-ESM2-1
  Cells   : 26  |  Time steps: 40,177
  Saved  : YARMOUK_JORDAN_3model_ensemble_ssp245.nc  (3.4 MB)  [3s]

[PROCESSING] J.VALLEY-YARMOUK TRIANGLE
  Models  : NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
  Cells   : 3  |  Time steps: 40,177
  Saved  : J.VALLEY-YARMOUK_TRIANGLE_3model_ensemble_ssp245.nc  (0.7 MB)  [2s]

[PROCESSING] JORDAN VALLY (JORDAN)
  Models  : MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
  Cells   : 21  |  Time steps: 40,177
  Saved  : JORDAN_VALLY_JORDAN_3model_ensemble_ssp245.nc  (2.5 MB)  [2s]

[PROCESSING] N.R.S.W
  Model

## 9. Verify Output Files

Spot-check the generated NetCDF files to confirm correct dimensions, coordinate ranges, variable attributes, and ensemble metadata.


In [10]:
nc_files = sorted(OUTPUT_DIR.glob("*_3model_ensemble_ssp245.nc"))
print(f"NetCDF files in output directory : {len(nc_files)}")
print()

for nc_f in nc_files:
    with xr.open_dataset(nc_f) as ds_v:
        pr = ds_v[PR_VAR]
        print(f"{'='*65}")
        print(f"File     : {nc_f.name}")
        print(f"Basin    : {ds_v.attrs.get('basin', '?')}")
        print(f"Models   : {ds_v.attrs.get('source_models', '?')}")
        print(f"Time     : {len(ds_v.time):,} steps  "
              f"({pd.Timestamp(ds_v.time.values[0]).date()} → "
              f"{pd.Timestamp(ds_v.time.values[-1]).date()})")
        print(f"Lat      : {float(ds_v.lat.min()):.2f}° – {float(ds_v.lat.max()):.2f}°N  "
              f"({len(ds_v.lat)} pts)")
        print(f"Lon      : {float(ds_v.lon.min()):.2f}° – {float(ds_v.lon.max()):.2f}°E  "
              f"({len(ds_v.lon)} pts)")
        print(f"pr range : {float(pr.min()):.4f} – {float(pr.max()):.4f} mm/day")
        print(f"pr mean  : {float(pr.mean()):.4f} mm/day")
        print(f"Size     : {nc_f.stat().st_size/1024/1024:.1f} MB")
print()


NetCDF files in output directory : 16

File     : AMMAN_ZARQA_JORDAN_3model_ensemble_ssp245.nc
Basin    : AMMAN ZARQA (JORDAN)
Models   : MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
Time     : 40,177 steps  (1961-01-01 → 2070-12-31)
Lat      : 31.85° – 32.35°N  (6 pts)
Lon      : 35.65° – 36.65°E  (11 pts)
pr range : 0.0000 – 57.0788 mm/day
pr mean  : 0.6293 mm/day
Size     : 5.8 MB
File     : AZRAQ_JORDAN_3model_ensemble_ssp245.nc
Basin    : AZRAQ (JORDAN)
Models   : MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
Time     : 40,177 steps  (1961-01-01 → 2070-12-31)
Lat      : 31.05° – 32.55°N  (16 pts)
Lon      : 36.05° – 37.65°E  (17 pts)
pr range : 0.0000 – 57.0788 mm/day
pr mean  : 0.3387 mm/day
Size     : 16.7 MB
File     : D.S.R.S.W_3model_ensemble_ssp245.nc
Basin    : D.S.R.S.W
Models   : IPSL-CM6A-LR | MPI-ESM1-2-LR | NorESM2-MM
Time     : 40,177 steps  (1961-01-01 → 2070-12-31)
Lat      : 31.05° – 31.85°N  (9 pts)
Lon      : 35.35° – 35.85°E  (6 pts)
pr range : 0.0000 – 66.9233 mm/day


## 10. Summary Table

Final summary of all generated ensemble files, including the top-3 model assignments and number of grid cells per basin.


In [11]:
summary_rows = []
for basin_name, assign_info in basin_top3_assignment.items():
    mask_info = basin_masks.get(basin_name, {"n_cells": 0})
    nc_f      = OUTPUT_DIR / (safe_filename(basin_name) + "_3model_ensemble_ssp245.nc")
    exists    = nc_f.exists()
    size_mb   = round(nc_f.stat().st_size / 1024 / 1024, 1) if exists else np.nan
    top3      = assign_info["top3"]
    summary_rows.append({
        "Basin"         : basin_name,
        "Assign_Source" : assign_info["source"],
        "Rank1_Model"   : top3[0] if len(top3) > 0 else "-",
        "Rank2_Model"   : top3[1] if len(top3) > 1 else "-",
        "Rank3_Model"   : top3[2] if len(top3) > 2 else "-",
        "N_Grid_Cells"  : mask_info["n_cells"],
        "File_Exists"   : "YES" if exists else "NO",
        "Size_MB"       : size_mb,
    })

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "ensemble_basin_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("BASIN ENSEMBLE SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False))
print()
print(f"Summary saved: {summary_csv}")
print()

# Output directory tree
print("=" * 70)
print("OUTPUT DIRECTORY")
print("=" * 70)
for f in sorted(OUTPUT_DIR.iterdir()):
    sz = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<55}  {sz:>7.1f} MB")


BASIN ENSEMBLE SUMMARY
                    Basin                       Assign_Source   Rank1_Model   Rank2_Model   Rank3_Model  N_Grid_Cells File_Exists  Size_MB
                   HAMMAD                              direct MPI-ESM1-2-LR  CMCC-CM2-SR5    NorESM2-MM           217         YES     24.5
         YARMOUK (JORDAN)                              direct MPI-ESM1-2-LR    NorESM2-MM   CNRM-ESM2-1            26         YES      3.4
J.VALLEY-YARMOUK TRIANGLE         KD-tree → N.R.S.W (29.2 km)    NorESM2-MM   CNRM-ESM2-1  CMCC-CM2-SR5             3         YES      0.7
    JORDAN VALLY (JORDAN)                              direct MPI-ESM1-2-LR  CMCC-CM2-SR5    NorESM2-MM            21         YES      2.5
                  N.R.S.W                              direct    NorESM2-MM   CNRM-ESM2-1  CMCC-CM2-SR5            16         YES      2.0
           AZRAQ (JORDAN)                              direct MPI-ESM1-2-LR    NorESM2-MM  CMCC-CM2-SR5           149         YES     16.7
    